# Adapting DeepGaze MSDB to a new dataset

DeepGaze MSDB carries a small set of **per-dataset parameters** (13 scalars: multi-scale weights, a gaussian blur sigma, a center-bias weight and a priority scaling) on top of a frozen CLIP+DINOv2 backbone and a shared saliency network. To adapt the model to a *new* dataset we add one new parameter slot and train only those 13 scalars.

The new slot is initialised from the average of the model's original datasets, so it starts from the model's generalization behaviour (`dataset=None`) and only needs a little data to specialise. Adapting does **not** change the predictions for the built-in datasets.

This notebook adapts the released model to the public **Toronto** dataset. Replace the data-loading cell with your own `images/` folder and a `fixations.csv` (columns `image,x,y`) to adapt to your data.

In [ ]:
import numpy as np
import torch

from deepgaze_pytorch import DeepGazeMSDB, MSDBDataset
from deepgaze_pytorch.custom_data import load_fixations_csv, fit_centerbias
from deepgaze_pytorch.msdb_adaptation import adapt_dataset_parameters

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PIXEL_PER_DVA = 21.75   # pixels per degree of visual angle of your presentation (Toronto ~21.75)

## 1. Load your data

`load_fixations_csv` turns an image folder + a fixation CSV into pysaliency `stimuli, fixations`. If you already have pysaliency objects, skip this and use them directly. Here we build such a CSV from the Toronto dataset shipped with pysaliency, purely to have a concrete public example.

In [ ]:
import csv, os
import pysaliency

# Get Toronto from pysaliency (downloads on first use) and write an images-folder + CSV,
# so this example uses exactly the same public entry point as your own data would.
toronto_stimuli, toronto_fixations = pysaliency.external_datasets.get_toronto(location='pysaliency_datasets')
image_dir = os.path.join('pysaliency_datasets', 'toronto', 'stimuli')
os.makedirs('adaptation_example', exist_ok=True)
csv_path = os.path.join('adaptation_example', 'toronto_fixations.csv')
names = [os.path.basename(f) for f in toronto_stimuli.filenames]
with open(csv_path, 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['image', 'x', 'y'])
    for x, y, n in zip(toronto_fixations.x, toronto_fixations.y, toronto_fixations.n):
        w.writerow([names[int(n)], float(x), float(y)])

# --- this is the part you would run on your own data: ---
stimuli, fixations = load_fixations_csv(image_dir, csv_path)
print(len(stimuli), 'images,', len(fixations), 'fixations')

## 2. Fit a center bias

DeepGaze models take a center-bias (a baseline log-density of where fixations land, independent of the image) as input. `fit_centerbias` fits one over your fixations, choosing the KDE bandwidth by leave-one-image-out cross-validation. Pass your own model instead if you have one.

In [ ]:
centerbias = fit_centerbias(stimuli, fixations, verbose=True)

## 2b. Hold out images for evaluation

To see what adaptation brings for images it has not seen, we adapt on one half of the images and evaluate on the other half. The center bias above is leave-one-image-out crossvalidated, so it can serve both halves: the fixations of an image never enter its own prior.

In [ ]:
from pysaliency.datasets import create_subset

image_order = np.random.RandomState(0).permutation(len(stimuli))
adapt_stimuli, adapt_fixations = create_subset(stimuli, fixations, np.sort(image_order[::2]))
test_stimuli, test_fixations = create_subset(stimuli, fixations, np.sort(image_order[1::2]))
print(len(adapt_stimuli), 'adaptation images,', len(test_stimuli), 'held-out images')

## 3. Load the released model and add a dataset slot

`add_dataset()` appends the new slot and returns its index. As a sanity check, at initialisation the new slot reproduces the averaged `dataset=None` prediction exactly.

In [ ]:
model = DeepGazeMSDB(pretrained=True).to(DEVICE).eval()

def predict(stimulus, dataset):
    cb = centerbias.log_density(stimulus)
    img = torch.tensor(np.array([np.asarray(stimulus).transpose(2, 0, 1)]), dtype=torch.float32, device=DEVICE)
    cbt = torch.tensor(np.array([cb]), dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        return model(img, cbt, pixel_per_dva=PIXEL_PER_DVA, dataset=dataset)[0].cpu().numpy()

probe = stimuli.stimuli[0]
averaged = predict(probe, None)
dataset_index = model.add_dataset()
print('new dataset index:', dataset_index)
print('init reproduces dataset=None, max abs diff:', np.abs(predict(probe, dataset_index) - averaged).max())

## 4. Adapt (train the 13 parameters)

`adapt_dataset_parameters` trains only the new slot, on the adaptation images. We pass the `dataset_index` we just created so it does not add a second slot. The held-out images are passed as validation data only for logging: the adapted parameters are the ones after the last epoch, so the held-out images do not influence the fit.

The run is saved under `train_directory`, so **re-running this cell is cheap**: once training has finished it returns instantly with the adapted weights loaded, and an interrupted run resumes from its last checkpoint — handy for experimenting without retraining each time. (Each call adds a new slot, so re-run the model-construction cell above too, or the already-adapted model would be widened a second time.)

In [ ]:
model, dataset_index = adapt_dataset_parameters(
    model, adapt_stimuli, adapt_fixations, centerbias, PIXEL_PER_DVA,
    val_stimuli=test_stimuli, val_fixations=test_fixations,
    dataset_index=dataset_index, train_directory='adaptation_example/run_holdout',
)

## 5. Compare zero-shot vs adapted

We score information gain (bit/fixation over the center bias, image-averaged) on the **held-out** images for the averaged (`dataset=None`) model and for the adapted slot. Scoring on the adaptation images instead would measure the fit, not what adaptation brings for new images.

In [ ]:
class _MSDB(pysaliency.Model):
    def __init__(s, dataset): super().__init__(caching=False); s.dataset = dataset
    def _log_density(s, stim): return predict(pysaliency.datasets.as_stimulus(stim), s.dataset)

for name, dataset in [('zero-shot (dataset=None)', None), ('adapted', dataset_index)]:
    ig = _MSDB(dataset).information_gain(test_stimuli, test_fixations, centerbias, average='image')
    print(f'{name:26s} IG = {ig:.4f} bit/fix (held-out images)')

## Notes and next steps

- On Toronto the gain from adaptation is small — the averaged parameters already generalize well. Other datasets may benefit more.
- The adapted model still serves the built-in datasets (`dataset=MSDBDataset.MIT1003`, ...) unchanged, and you can adapt to several datasets in turn.
- Save the adapted (head-only) weights with `torch.save(model.head_state_dict(), 'adapted.pth')`. To reload: build a `DeepGazeMSDB`, call `model.add_dataset()`, then `model.load_state_dict(torch.load('adapted.pth'), strict=False)`.
- **Ideas to extend this notebook:** visualise the learned per-dataset parameters before vs after adaptation, or plot a few predicted density maps against the ground-truth fixations.